<a href="https://colab.research.google.com/github/Pooja-V15/NVIDIA-AI-ML-Internship/blob/main/Day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# TOPIC 1 - PROMPT ENGINEERING
# Same task asked three ways, so students see better prompts give better answers.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

def ask(text, n=120):
    msg = [{"role": "user", "content": text}]
    p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=n, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Zero-shot: just ask
print("Zero-shot:", ask("Is this review positive or negative? 'Broke in two days.'", 10))

# Few-shot: show the pattern first
print("Few-shot :", ask("good->Positive  bad->Negative  'fast and reliable'->", 5))

# Chain-of-thought: ask it to think step by step (gives smarter answers)
print("Step-by-step:", ask("Solve step by step: a train goes 60km in 1.5h. Average speed?"))


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Zero-shot: The review "Broke in two days." is
Few-shot : "Fast and reliable"
Step-by-step: To find the average speed of the train, we use the formula for average speed:

\[
\text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}}
\]

Given:
- Total Distance \(d\) = 60 km
- Total Time \(t\) = 1.5 hours

Now, plug these values into the formula:

\[
\text{Average Speed} = \frac{60 \text{ km}}{1.5 \text{ h}}
\]

Perform the division:

\[
\text{Average Speed}


In [2]:
# TOPIC 2 - HUGGING FACE ON NVIDIA H200
# Load a model on the GPU and see how much memory it uses.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Check the GPU (on the lab box this shows the H200 with ~141 GB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

# How much GPU memory the model takes
if torch.cuda.is_available():
    print("Model uses about", round(torch.cuda.memory_allocated()/1e9, 2), "GB of GPU memory")

# Generate one reply
msg = [{"role": "user", "content": "Explain what a GPU does in one sentence."}]
p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
ids = tok(p, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=60, pad_token_id=tok.eos_token_id)
print("Reply:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

# The H200's big 141 GB memory is what lets you run bigger models and serve
# many users at once without running out of space.


GPU: Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model uses about 3.1 GB of GPU memory
Reply: A GPU (Graphics Processing Unit) is designed to perform intensive mathematical and computational tasks for rendering graphics and images efficiently.


In [3]:
pip install trl

In [4]:
pip install -U bitsandbytes>=0.46.1

In [5]:
print('Uninstalling previous bitsandbytes versions...')
!pip uninstall -y bitsandbytes
print('Installing compatible bitsandbytes, accelerate, peft, and trl versions...')
!pip install -U bitsandbytes>=0.46.1 accelerate peft trl

Uninstalling previous bitsandbytes versions...
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Installing compatible bitsandbytes, accelerate, peft, and trl versions...


In [6]:
print('Updating transformers...')
!pip install -U transformers

Updating transformers...


In [7]:
print('Updating huggingface_hub...')
!pip install -U huggingface_hub

Updating huggingface_hub...


### Hugging Face Token Configuration

YouThe warning about `HF_TOKEN` indicates that you are not authenticated with the Hugging Face Hub. While this is often optional for public models, authenticating can prevent rate limits and allow you to access private models or upload your own.

To resolve this, you should:

1.  **Generate a Hugging Face Token**: Go to [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a new access token (with 'write' role if you plan to upload models).
2.  **Add to Colab Secrets**: In Google Colab, click the '🔑' icon (Secrets) in the left sidebar. Add a new secret named `HF_TOKEN` and paste your Hugging Face token as its value.
3.  **Enable access**: Ensure 'Notebook access' is toggled on for your notebook.

After setting the secret, restart your runtime and re-run the cells.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token

# Load the model in 4-bit so it fits easily (the "Q" in QLoRA)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16) # Explicitly set torch_dtype to float16
model = prepare_model_for_kbit_training(model)

# Attach small LoRA adapters (the sticky notes); only these get trained
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                       target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()   # shows under 1% of parameters are trained

# A small set of example instructions to learn from
data = load_dataset("yahma/alpaca-cleaned", split="train[:200]").map(
    lambda e: {"text": f"### Instruction:\n{e['instruction']}\n\n### Response:\n{e['output']}"})

# Train, then save the tiny adapter (only a few MB)
SFTTrainer(model=model, train_dataset=data, processing_class=tok,
    args=SFTConfig(output_dir="./out", num_train_epochs=1, per_device_train_batch_size=4,
                   learning_rate=2e-4, bf16=False, fp16=False, # Disabled fp16 and bf16 to avoid torch.amp.grad_scaler issues
                   dataset_text_field="text", report_to="none")).train()
model.save_pretrained("./day7-adapter")
print("Done - adapter saved.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.049113
20,1.695251
30,1.622236
40,1.731354
50,1.237809


Done - adapter saved.


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token

# Load the model in 4-bit so it fits easily (the "Q" in QLoRA)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model)

# Attach small LoRA adapters (the sticky notes); only these get trained
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                       target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()   # shows under 1% of parameters are trained

# A small set of example instructions to learn from
data = load_dataset("yahma/alpaca-cleaned", split="train[:200]").map(
    lambda e: {"text": f"### Instruction:\n{e['instruction']}\n\n### Response:\n{e['output']}"})

# Train, then save the tiny adapter (only a few MB)
SFTTrainer(model=model, train_dataset=data, processing_class=tok,
    args=SFTConfig(output_dir="./out", num_train_epochs=1, per_device_train_batch_size=4,
                   learning_rate=2e-4, bf16=False, fp16=False, # Disabled bf16 and fp16 to avoid torch.amp.grad_scaler issues
                   dataset_text_field="text", report_to="none")).train()
model.save_pretrained("./day7-adapter")
print("Done - adapter saved.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.046607
20,1.693871
30,1.622190
40,1.733058
50,1.237239


Done - adapter saved.


In [8]:
# TOPIC 3 (part 2) - USE THE TRAINED ADAPTER
# Run this after 3_lora_finetuning has saved ./day7-adapter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

# Load the big base model, then stick the tiny trained adapter on top
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, "./day7-adapter")

# Ask it something
prompt = "### Instruction:\nExplain what a GPU does in one sentence.\n\n### Response:\n"
ids = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=80, pad_token_id=tok.eos_token_id)
print("Reply:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

# Optional: merge the adapter into the base model for fast deployment
merged = model.merge_and_unload()
print("Adapter merged into the model - ready to deploy.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reply: A GPU, or graphics processing unit, is a specialized processor designed to accelerate the performance of complex calculations and operations for tasks such as rendering images, video games, and scientific simulations.
Adapter merged into the model - ready to deploy.


In [7]:
print('Updating torchao...')
!pip install -U torchao>=0.16.0

Updating torchao...
